In [1]:
"""
exp01 Step 04: Multivariate Signal Check (DIAGNOSTIC, PROTOCOL.md §13.3; protocol 1.1, decided by Nicolas 2026-10-06)

exp01 is STOPPED (step 02 verdict, p = 0.10). This step was added AFTER exp01's results were known: it can explain the
failure; it CANNOT reverse the STOP or restart exp01's walk-forward.

Question: (a) does the COMBINATION of the 33 model features, through exp01's own LightGBM classifiers (one per distance,
fit on training only), predict "take profit first" better than the base rate on the pooled validation rows, beyond what
chance and the model's flexibility produce? (b) If so, does the top decile of predicted P(TP) clear its break-even rate?

    - Windows, sampling, features and distances exactly as step 02 (asserted against its trial-log entry).
    - Real check: AUC, top-decile TP rate, lift and break-even per distance, block-bootstrap intervals of ISO weeks.
    - Null: the feature columns shifted by whole sessions (step 02's shift), the models RETRAINED, 19 runs.
    - Verdict: INFORMATION only if the real mean AUC over the 21 distances is above EVERY null run (p <= 1/20; ties: none).
      Outcome A (no information), B (information, not tradable), C (information, possibly tradable: candidate pending
      review, the untouched-window decision is parked, no trading experiment is built on it).
    - Nothing on or after 2026-05-14 is loaded (raw bars cut at the last validation date; every row asserted).
One trial-log entry (stage "multivariate_check"). Prerequisite: exp01 step 01. Outputs:
store04_experiments/exp01_minute_entry/step04_multivariate_check_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import json
import time
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT RAW DATA, TRADE EXECUTION AND FILE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.core.local_file_management import get_date_range_file_path_list, write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import get_model_feature_col_str_list
from experiments.exp01_minute_entry.multivariate_check import get_schedule_tuple, get_check_window_dict, read_check_window_pdf_tuple, check_last_date_bool, \
                                                               get_check_lgbm_param_dict, run_model_check_dict, get_null_run_pdf_tuple, \
                                                               get_multivariate_verdict_dict, get_calibration_pdf, get_feature_importance_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
print("exp01 status: STOPPED (step 02, p = 0.10). Step 04 is a diagnostic added after exp01's results: it cannot reverse the STOP.")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step04_multivariate_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, exp_config.MULTIVARIATE_CHECK_STAGE_STR)

Experiment:	exp01_minute_entry	Protocol:	1.1	Config Hash:	1ea6cfdc0a
exp01 status: STOPPED (step 02, p = 0.10). Step 04 is a diagnostic added after exp01's results: it cannot reverse the STOP.
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/
ℹ️ No earlier 'multivariate_check' entry of exp01_minute_entry in the trial log (config hash 1ea6cfdc0a).


In [4]:
# COLLECT THE SESSION DATES FROM THE MODEL DATASET FILE NAMES (THE SCHEDULE IS ANCHORED ON THE LAST SESSION, AS STEP 02; NO FILE IS READ HERE)
dataset_file_path_list = get_date_range_file_path_list(paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data"))
session_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in dataset_file_path_list]
# CALL FUNCTION TO GET THE FOLDS AND THE POOLED VALIDATION FOLDS
fold_pdf, pooled_fold_pdf = get_schedule_tuple(session_date_list)
window_dict = get_check_window_dict(pooled_fold_pdf)
# DISPLAY INFORMATION
print(f"Train:\t\t{window_dict['train_start']} -> {window_dict['train_end']}\t({window_dict['train_years']} years)")
for _, row in pooled_fold_pdf.iterrows():
    print(f"Validation:\t{row['valid_start']} -> {row['valid_end']}\t(fold {row['fold_id']})")
print(f"Latest test window (not read):\t{fold_pdf['test_start'].iloc[-1]} -> {fold_pdf['test_end'].iloc[-1]}")
# ASSERT THE WINDOWS ARE THOSE OF STEP 02 (ITS CALIBRATED TRIAL-LOG ENTRY, HASH 15716b01a3)
trial_log_pdf = read_trial_log_pdf()
step02_pdf = trial_log_pdf[(trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME) & (trial_log_pdf["stage"] == "signal_check") &
                           (trial_log_pdf["config_hash"] == "15716b01a3")] if not trial_log_pdf.empty else pd.DataFrame()
if len(step02_pdf):
    step02_setting_dict = json.loads(step02_pdf.iloc[-1]["setting_json"])
    assert str(step02_setting_dict["train_start"]) == str(window_dict["train_start"]) and str(step02_setting_dict["train_end"]) == str(window_dict["train_end"])
    assert step02_setting_dict["valid_fold_id_list"] == window_dict["valid_fold_id_list"]
    print(f"✅ Windows identical to step 02 (trial {step02_pdf.iloc[-1]['trial_id']})")
else:
    print("⚠️ No step 02 entry with hash 15716b01a3 in this trial log (smoke run only): windows not compared")
# ASSERT THE LAST VALIDATION DATE IS BEFORE THE UNTOUCHED WINDOW
last_valid_date = window_dict["valid_end"]
assert pd.Timestamp(last_valid_date) < pd.Timestamp(exp_config.MULTIVARIATE_CHECK_UNTOUCHED_START_DATE_STR)
print(f"Last date loaded:\t{last_valid_date}\t(untouched window starts {exp_config.MULTIVARIATE_CHECK_UNTOUCHED_START_DATE_STR})")

Train:		2015-04-16 -> 2025-04-15	(10 years)
Validation:	2025-05-01 -> 2025-07-30	(fold 41)
Validation:	2025-07-31 -> 2025-10-30	(fold 42)
Validation:	2025-10-31 -> 2026-01-30	(fold 43)
Validation:	2026-01-30 -> 2026-04-29	(fold 44)
Latest test window (not read):	2026-05-14 -> 2026-08-13
✅ Windows identical to step 02 (trial 20261006124646690847_15716b01a3)
Last date loaded:	2026-04-29	(untouched window starts 2026-05-14)


In [5]:
# CALL FUNCTION TO READ THE TRAINING AND POOLED VALIDATION ROWS (AS STEP 02; SORTED BY DECISION TIME)
train_pdf, valid_pdf = read_check_window_pdf_tuple(window_dict, pooled_fold_pdf, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO READ THE RAW BARS UP TO THE LAST VALIDATION DATE (UNIQUENESS WEIGHTS OF THE TRAINING ROWS)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=str(last_valid_date))
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# ASSERT NO ROW AND NO BAR IS DATED AFTER THE LAST VALIDATION DATE
assert check_last_date_bool([train_pdf["decision_ts"], valid_pdf["decision_ts"], complete_ohlcv_pdf["timestamp"]], last_valid_date)
# DEFINE THE FEATURE LIST (ABSOLUTE FEATURES EXCLUDED, AS STEPS 02-03)
feature_col_str_list = get_model_feature_col_str_list(train_pdf, exp_config.MODEL_FEATURE_SCALE_TYPE_LIST)
# DISPLAY INFORMATION
print(f"Train Rows:\t{len(train_pdf):,}\tPooled Validation Rows:\t{len(valid_pdf):,}\t(step 02: 60,144 and 5,964)")
print(f"Rows dated:\t{train_pdf['decision_ts'].min()} -> {valid_pdf['decision_ts'].max()}\tBars:\t{complete_ohlcv_pdf['timestamp'].min()} -> {complete_ohlcv_pdf['timestamp'].max()}")
print(f"Features ({len(feature_col_str_list)}):\t{feature_col_str_list}")
print(f"Distances ({len(exp_config.SIGNAL_CHECK_DELTA_LIST)}):\t{exp_config.SIGNAL_CHECK_DELTA_LIST}")
print(f"Model:\t{get_check_lgbm_param_dict()}\tweights: {exp_config.SAMPLE_WEIGHT_MODE}")

Reading 2,516 model dataset files (2015-04-16 -> 2025-04-15, sampling: 'interval')


Train Rows:	60,144	Pooled Validation Rows:	5,964	(step 02: 60,144 and 5,964)
Rows dated:	2015-04-16 09:59:00-04:00 -> 2026-04-29 15:44:00-04:00	Bars:	2005-01-03 09:30:00-05:00 -> 2026-04-29 15:59:00-04:00
Features (33):	['trend', 'segment', 'color', 'bot_wick_pct', 'body_pct', 'top_wick_pct', 'stoch_k', 'stoch_d', 'rsi', 'bb_pct', 'dc_pct', 'ms_minima_trend', 'ms_maxima_trend', 'ms_low_status', 'ms_high_status', 'ms_trend', 'minima_count', 'maxima_count', 'prev_seg_delta_pct', 'seg_cs_count', 'seg', 'minutes_since_open', 'minutes_to_close', 'day_of_week', 'overnight_gap_pct', 'prev_day_return_pct', 'prev_day_range_pct', 'prev_day_high_dist_pct', 'prev_day_low_dist_pct', 'intraday_return_pct', 'daily_volatility', 'relative_volume', 'ath_drawdown_pct']
Distances (21):	[0.001, 0.0011, 0.0013, 0.0014, 0.0016, 0.0018, 0.002, 0.0022, 0.0025, 0.0028, 0.0032, 0.0035, 0.004, 0.0045, 0.005, 0.0056, 0.0063, 0.0071, 0.0079, 0.0089, 0.01]
Model:	{'n_estimators': 300, 'learning_rate': 0.03, 'num_lea

In [6]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO FIT THE REAL MODELS ON TRAINING AND MEASURE THEM ON THE POOLED VALIDATION (WITH INTERVALS)
real_dict = run_model_check_dict(train_pdf, valid_pdf, feature_col_str_list, ohlcv_array_dict, exp_config.SIGNAL_CHECK_DELTA_LIST)
metric_pdf = real_dict["metric_pdf"]
# DISPLAY INFORMATION
print(f"Real check processed in {(time.time() - start_time) / 60:.1f} minutes\tModels:\t{len(real_dict['model_dict'])}")
print(f"Real mean AUC over {len(metric_pdf)} distances:\t{real_dict['mean_auc']:.4f}")
# PREVIEW THE PER-DISTANCE MEASURES
metric_pdf.round(4)

Real check processed in 0.7 minutes	Models:	21
Real mean AUC over 21 distances:	0.4943


,delta,row_count,base_rate,auc,top_row_count,top_tp_rate,top_lift,top_breakeven_tp_rate,auc_ci_low,auc_ci_high,top_tp_rate_ci_low,top_tp_rate_ci_high,top_lift_ci_low,top_lift_ci_high,clears_breakeven
0,0.0010,5964,0.5055,0.5192,596,0.5403,0.0347,0.5190,0.4969,0.5432,0.4756,0.5995,-0.0212,0.0846,False
1,0.0011,5964,0.5042,0.5159,596,0.5235,0.0193,0.5173,0.4952,0.5396,0.4617,0.5844,-0.0312,0.0691,False
2,0.0013,5964,0.5047,0.5285,596,0.5436,0.0389,0.5147,0.5051,0.5554,0.4761,0.6142,-0.0169,0.0989,False
3,0.0014,5964,0.5013,0.5346,596,0.5671,0.0658,0.5137,0.5073,0.5620,0.4923,0.6416,-0.0004,0.1322,False
4,0.0016,5964,0.5023,0.5154,596,0.5302,0.0279,0.5119,0.4871,0.5449,0.4562,0.6054,-0.0352,0.0920,False
5,0.0018,5964,0.5017,0.5153,596,0.5319,0.0302,0.5106,0.4870,0.5458,0.4478,0.6137,-0.0402,0.0992,False
6,0.0020,5964,0.5039,0.5187,596,0.5319,0.0280,0.5096,0.4863,0.5519,0.4470,0.6198,-0.0479,0.1069,False
7,0.0022,5964,0.4997,0.5199,596,0.5336,0.0339,0.5087,0.4865,0.5531,0.4382,0.6198,-0.0472,0.1136,False
8,0.0025,5964,0.5023,0.5172,596,0.5302,0.0279,0.5077,0.4818,0.5519,0.4337,0.6171,-0.0549,0.1056,False
9,0.0028,5964,0.4995,0.4954,596,0.5084,0.0089,0.5068,0.4621,0.5271,0.4233,0.5902,-0.0696,0.0801,False


In [7]:
# RECORD THE START TIME
start_time = time.time()
# CALL FUNCTION TO RUN THE NULL CHECKS (FEATURES SHIFTED BY WHOLE SESSIONS, MODELS RETRAINED; ALL CORES, 1 LIGHTGBM THREAD PER MODEL)
print(f"Running {exp_config.SIGNAL_CHECK_NULL_RUN_COUNT} null checks in parallel ...")
null_summary_pdf, null_metric_pdf = get_null_run_pdf_tuple(train_pdf, valid_pdf, feature_col_str_list, ohlcv_array_dict, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO GET THE VERDICT
verdict_dict = get_multivariate_verdict_dict(metric_pdf, null_summary_pdf)
# DISPLAY INFORMATION
print(f"Null checks processed in {(time.time() - start_time) / 60:.1f} minutes\n")
for key_str, value in verdict_dict.items():
    print(f"{key_str}:\t{value}")
print(f"\nVERDICT:\tOutcome {verdict_dict['outcome']} ({verdict_dict['outcome_str']})\tp = {verdict_dict['null_p_value']:.2f}")
print("Reminder: exp01 stays STOPPED whatever this outcome (diagnostic added after its results).")
# PREVIEW THE NULL RUNS
null_summary_pdf.round(4)

Running 19 null checks in parallel ...


Null checks processed in 2.4 minutes

real_mean_auc:	0.494252977966177
null_run_count:	19
null_mean_auc_list:	[0.5050495600004608, 0.4960489246593752, 0.4876333662061407, 0.49593423750322063, 0.5057524760217981, 0.5258800449013931, 0.49889168420658936, 0.5048557104519213, 0.5047317794599581, 0.4869415004239438, 0.5282285435839692, 0.4850684551533975, 0.5215184801811653, 0.5151338952069529, 0.501385969633535, 0.5044956484546061, 0.4867393059901592, 0.49457294777457744, 0.4693728518571092]
null_max_mean_auc:	0.5282285435839692
null_p_value:	0.75
information:	False
clears_breakeven_delta_list:	[]
outcome:	A
outcome_str:	no information

VERDICT:	Outcome A (no information)	p = 0.75
Reminder: exp01 stays STOPPED whatever this outcome (diagnostic added after its results).


,run_id,train_shift_session_count,valid_shift_session_count,mean_auc,mean_top_lift
0,0,741,159,0.5050,0.0306
1,1,1432,161,0.4960,0.0073
2,2,908,98,0.4876,-0.0352
3,3,1603,89,0.4959,-0.0338
4,4,1132,156,0.5058,-0.0330
5,5,957,84,0.5259,0.0620
6,6,1482,111,0.4989,0.0063
7,7,1234,127,0.5049,0.0084
8,8,1143,176,0.5047,0.0342
9,9,1639,127,0.4869,0.0126


In [8]:
# COMPARE EACH DISTANCE WITH ITS NULL DISTRIBUTION (DESCRIPTIVE, NOT PART OF THE VERDICT)
null_stat_pdf = null_metric_pdf.groupby("delta").agg(null_auc_mean=("auc", "mean"), null_auc_max=("auc", "max"), null_top_lift_mean=("top_lift", "mean"),
                                                     null_top_lift_max=("top_lift", "max")).reset_index()
distance_pdf = metric_pdf.merge(null_stat_pdf, on="delta", how="left")
distance_pdf["null_auc_ge_real_count"] = [int((null_metric_pdf.loc[null_metric_pdf["delta"] == row["delta"], "auc"] >= row["auc"]).sum()) for _, row in distance_pdf.iterrows()]
distance_pdf["null_top_lift_ge_real_count"] = [int((null_metric_pdf.loc[null_metric_pdf["delta"] == row["delta"], "top_lift"] >= row["top_lift"]).sum())
                                               for _, row in distance_pdf.iterrows()]
# DISPLAY THE PER-DISTANCE TABLE (REAL VS NULL; TOP DECILE VS BREAK-EVEN)
display(distance_pdf[["delta", "row_count", "base_rate", "auc", "auc_ci_low", "auc_ci_high", "null_auc_mean", "null_auc_max", "null_auc_ge_real_count"]].round(4))
distance_pdf[["delta", "top_row_count", "top_tp_rate", "top_tp_rate_ci_low", "top_tp_rate_ci_high", "top_breakeven_tp_rate", "clears_breakeven",
              "top_lift", "top_lift_ci_low", "top_lift_ci_high", "null_top_lift_mean", "null_top_lift_max", "null_top_lift_ge_real_count"]].round(4)

,delta,row_count,base_rate,auc,auc_ci_low,auc_ci_high,null_auc_mean,null_auc_max,null_auc_ge_real_count
0,0.0010,5964,0.5055,0.5192,0.4969,0.5432,0.4984,0.5147,0
1,0.0011,5964,0.5042,0.5159,0.4952,0.5396,0.4975,0.5140,0
2,0.0013,5964,0.5047,0.5285,0.5051,0.5554,0.4993,0.5145,0
3,0.0014,5964,0.5013,0.5346,0.5073,0.5620,0.4979,0.5148,0
4,0.0016,5964,0.5023,0.5154,0.4871,0.5449,0.4985,0.5167,1
5,0.0018,5964,0.5017,0.5153,0.4870,0.5458,0.4997,0.5276,4
6,0.0020,5964,0.5039,0.5187,0.4863,0.5519,0.5028,0.5305,4
7,0.0022,5964,0.4997,0.5199,0.4865,0.5531,0.5008,0.5264,3
8,0.0025,5964,0.5023,0.5172,0.4818,0.5519,0.5013,0.5411,6
9,0.0028,5964,0.4995,0.4954,0.4621,0.5271,0.5006,0.5488,13


,delta,top_row_count,top_tp_rate,top_tp_rate_ci_low,top_tp_rate_ci_high,top_breakeven_tp_rate,clears_breakeven,top_lift,top_lift_ci_low,top_lift_ci_high,null_top_lift_mean,null_top_lift_max,null_top_lift_ge_real_count
0,0.0010,596,0.5403,0.4756,0.5995,0.5190,False,0.0347,-0.0212,0.0846,-0.0015,0.0381,2
1,0.0011,596,0.5235,0.4617,0.5844,0.5173,False,0.0193,-0.0312,0.0691,-0.0059,0.0394,2
2,0.0013,596,0.5436,0.4761,0.6142,0.5147,False,0.0389,-0.0169,0.0989,-0.0058,0.0389,1
3,0.0014,596,0.5671,0.4923,0.6416,0.5137,False,0.0658,-0.0004,0.1322,-0.0042,0.0440,0
4,0.0016,596,0.5302,0.4562,0.6054,0.5119,False,0.0279,-0.0352,0.0920,-0.0073,0.0564,3
5,0.0018,596,0.5319,0.4478,0.6137,0.5106,False,0.0302,-0.0402,0.0992,-0.0003,0.0537,5
6,0.0020,596,0.5319,0.4470,0.6198,0.5096,False,0.0280,-0.0479,0.1069,0.0060,0.0867,5
7,0.0022,596,0.5336,0.4382,0.6198,0.5087,False,0.0339,-0.0472,0.1136,-0.0042,0.0725,5
8,0.0025,596,0.5302,0.4337,0.6171,0.5077,False,0.0279,-0.0549,0.1056,0.0043,0.0748,9
9,0.0028,596,0.5084,0.4233,0.5902,0.5068,False,0.0089,-0.0696,0.0801,0.0022,0.0928,7


In [9]:
# CALL FUNCTION TO GET THE CALIBRATION TABLE (DECILES OF PREDICTED P(TP) PER DISTANCE, POOLED; DESCRIPTIVE)
calibration_pdf = get_calibration_pdf(valid_pdf, real_dict["proba_pdf"], exp_config.SIGNAL_CHECK_DELTA_LIST)
display(calibration_pdf.round(4))
# CALL FUNCTION TO GET THE SPLIT-GAIN IMPORTANCE OF THE REAL MODELS (DESCRIPTIVE: WHAT THE MODELS USED, NOT EVIDENCE OF INFORMATION)
importance_pdf = get_feature_importance_pdf(real_dict["model_dict"], feature_col_str_list)
print(f"Top {exp_config.MULTIVARIATE_CHECK_IMPORTANCE_TOP_COUNT} features by mean split-gain share over the {len(real_dict['model_dict'])} models (descriptive):")
importance_pdf.head(exp_config.MULTIVARIATE_CHECK_IMPORTANCE_TOP_COUNT).round(4)

,decile,row_count,mean_predicted,observed_tp_rate,gap
0,1,12537,0.4156,0.5076,0.0920
1,2,12516,0.4571,0.5058,0.0487
2,3,12516,0.4765,0.5002,0.0238
3,4,12537,0.4910,0.5011,0.0101
4,5,12516,0.5035,0.4850,-0.0185
5,6,12516,0.5153,0.4845,-0.0308
6,7,12537,0.5274,0.4913,-0.0362
7,8,12516,0.5409,0.5042,-0.0368
8,9,12516,0.5586,0.5018,-0.0568
9,10,12537,0.5974,0.4822,-0.1153


Top 10 features by mean split-gain share over the 21 models (descriptive):


,feature,mean_gain_share,mean_gain
0,prev_day_return_pct,0.0944,11769.3694
1,overnight_gap_pct,0.0914,11218.0337
2,prev_day_range_pct,0.0820,10444.5671
3,daily_volatility,0.0808,10115.3492
4,intraday_return_pct,0.0739,7904.3704
5,ath_drawdown_pct,0.0714,8583.7084
6,prev_day_high_dist_pct,0.0610,6919.0763
7,prev_day_low_dist_pct,0.0535,5768.0391
8,prev_seg_delta_pct,0.0393,3896.3299
9,day_of_week,0.0354,4535.2428


In [10]:
# SAVE THE RESULTS
write_csv_file_to_path(distance_pdf, f"{output_path_str}distance_metric_data.csv", "W")
write_csv_file_to_path(null_summary_pdf, f"{output_path_str}null_run_data.csv", "W")
write_csv_file_to_path(null_metric_pdf, f"{output_path_str}null_distance_metric_data.csv", "W")
write_csv_file_to_path(calibration_pdf, f"{output_path_str}calibration_data.csv", "W")
write_csv_file_to_path(importance_pdf, f"{output_path_str}feature_importance_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/distance_metric_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/distance_metric_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/null_run_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/null_run_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step04_multivariate_check_data/null_distance_metric_data.csv' 

In [11]:
# DEFINE THE LOGGED WINDOWS (THE CHECK'S TRAINING AND POOLED VALIDATION; THE LATEST TEST WINDOW WAS NOT READ)
last_fold_row = pooled_fold_pdf.iloc[-1]
log_window_dict = {"fold_id": int(last_fold_row["fold_id"]), "train_start": window_dict["train_start"], "train_end": window_dict["train_end"],
                   "valid_start": window_dict["valid_start"], "valid_end": window_dict["valid_end"], "test_start": last_fold_row["test_start"], "test_end": last_fold_row["test_end"]}
# DEFINE THE PER-DISTANCE SUMMARIES
distance_summary_list = distance_pdf[["delta", "row_count", "base_rate", "auc", "auc_ci_low", "auc_ci_high", "null_auc_max", "top_tp_rate", "top_tp_rate_ci_low",
                                      "top_breakeven_tp_rate", "clears_breakeven", "top_lift", "top_lift_ci_low", "top_lift_ci_high"]].round(6).to_dict("records")
# LOG THE TRIAL (ONE ENTRY; THE NULL RUNS ARE PART OF IT)
multivariate_trial_dict = log_trial_dict(exp_config, exp_config.MULTIVARIATE_CHECK_STAGE_STR,
                                         {"train_years": window_dict["train_years"], "train_start": window_dict["train_start"], "train_end": window_dict["train_end"],
                                          "valid_fold_id_list": window_dict["valid_fold_id_list"], "valid_start": window_dict["valid_start"], "valid_end": window_dict["valid_end"],
                                          "sampling": [exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES], "feature_list": feature_col_str_list,
                                          "delta_list": exp_config.SIGNAL_CHECK_DELTA_LIST, "lgbm_param_dict": get_check_lgbm_param_dict(),
                                          "sample_weight_mode": exp_config.SAMPLE_WEIGHT_MODE, "null_run_count": exp_config.SIGNAL_CHECK_NULL_RUN_COUNT,
                                          "null_shift_share_range": exp_config.SIGNAL_CHECK_NULL_SHIFT_SHARE_RANGE, "null_seed": config.RANDOM_SEED,
                                          "top_share": exp_config.MULTIVARIATE_CHECK_TOP_SHARE, "bootstrap": [exp_config.SIGNAL_CHECK_BOOTSTRAP_BLOCK_STR, config.BOOTSTRAP_ITERATION_COUNT]},
                                         {**verdict_dict, "distance_summary_list": distance_summary_list}, False, log_window_dict,
                                         note_str_in="exp01 step04 multivariate signal check (diagnostic added 2026-10-06 after exp01's STOP; cannot reverse it)")
print(f"Logged 1 '{exp_config.MULTIVARIATE_CHECK_STAGE_STR}' entry at {multivariate_trial_dict['logged_ts']} (config hash {get_config_hash_str(exp_config)})")

Logged 1 'multivariate_check' entry at 2026-10-06T18:27:48 (config hash 1ea6cfdc0a)

In [12]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
print(f"Trial log rows:\t{len(trial_log_pdf):,}")
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

Trial log rows:	3,293


,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
